In [113]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [114]:
import pandas as pd
import numpy as np

In [115]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### The Network Name in db and Ted's spread

In [116]:
path = '/workspaces/crmprtd/update_sheet/'
df = pd.read_excel(path + 'Ted_suggested_network_name.xlsx', header=[0, 1, 2])

df_form = df['New Name']['PCIC PCDS Naming'][['Current', 'Suggested Network name (blanks infer no change)']]


# ['Current']

df_form_clean = df_form.dropna(subset=["Current"]).copy()
df_form_clean["Current"] = df_form_clean["Current"].str.strip()
df_form_clean = df_form_clean.rename(
    columns={"Suggested Network name (blanks infer no change)": "Suggested Network name"}
)

new_rows = pd.DataFrame([
    {"Current": "EC_raw", "Suggested Network name": "EC_Raw"},
    {"Current": "MoTIm", "Suggested Network name": "BC-TRAN"}
])

df_form_clean = pd.concat([df_form_clean, new_rows], ignore_index=True)

df_form_clean

,Current,Suggested Network name
0,AGRI,BC AGRIWeather
1,ENV-AQN,BC-Air
2,ENV-ASP,BC-Snow
3,FLNRO-WMB,BC-TS
4,FLNRO-FERN,BC-FERN
5,MoTIe,BC-TRAN
6,ARDA,ARDA
7,FRBC,FRBC
8,BCH,BCH-GSO-HMP
9,PC-Aval,PC-Aval


In [117]:

network_sql = sa.text("""
SELECT
    network_id,
    network_name,
    description
FROM meta_network
""")

with engine.begin() as conn:
        db_network = pd.read_sql(network_sql, conn)

db_network

,network_id,network_name,description
0,5,BCH,BC Hydro
1,13,MoTI,None
2,16,FRBC,Forest Renewal British Columbia
3,17,ARDA,Agricultural and Rural Development Act Network
4,22,ARD,Agricultural Resource Development office - Alb...
5,23,MoAg_GS,BC Ministry of Agriculture Growers Supply
6,35,UNBC_CAM,UNBC Cariboo Alpine Mesonet
7,37,YT_WFM,Yukon Wildland Fire Management Branch
8,38,YAA,Yukon Avalanche Association
9,39,DFO_CCG_lighthouse,Department of Fisheries and Oceans - Lighthouses


In [119]:
df_merged = db_network.merge(
    df_form_clean,
    left_on="network_name",
    right_on="Current",
    how="outer"
)
df_merged = df_merged[~df_merged["Current"].astype(str).str.strip().eq("Mvan")]
df_merged["network_id"] = pd.to_numeric(df_merged["network_id"], errors="coerce").astype("Int64")

def determine_status(row):
    network = row["network_name"]    
    network_id = row["network_id"]
    current = row["Current"]
    suggested = row["Suggested Network name"]

    # Case 1: unchanged
    if pd.notna(network) and network == current and network == suggested:
        return "unchanged"

    # Case 2: rename
    if pd.notna(network) and network == current and pd.notna(suggested) and suggested != network:
        return "Rename"

    if pd.isnull(current) and pd.isnull(suggested):
        return "Not referenced"

    if pd.isnull(network_id):
        return "new"

    # Default
    return ""

df_merged["status"] = df_merged.apply(determine_status, axis=1)


df_merged.loc[df_merged["network_name"] == "MVRD", "status"] = "delete"
df_merged.loc[df_merged["network_name"] == "MVan", ["Suggested Network name", "status"]] = ["MVRD", "Rename"]

df_merged.to_csv("Network_name_update_summary.csv", index=False)


In [120]:
BCH need to be divided to BCH-GSO-HMP and BCH-SiteC?

MVRD, do we need to devide it into WMVR-AIR and WMVR-WS?


Object `SiteC` not found.
Object `WS` not found.
